In [12]:
no_of_threads = "12"

import os
os.environ["VECLIB_MAXIMUM_THREADS"] = no_of_threads
os.environ["OMP_NUM_THREADS"] = no_of_threads


In [13]:


import torch
from torch import linalg as LA


import time
import statistics
from typing import Tuple
from typing_extensions import Annotated
import transformers
import lm_eval
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import platform
import subprocess
import time
from datasets import load_dataset

In [14]:
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

In [15]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"   

model_hf = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float32, device_map="cpu"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1410.54it/s]


In [16]:
def time_counter(no_of_threads: int, model, tokenizer, prompts: list,
                 results: list, max_new_tokens: int = 128, repeats: int = 3):
    

    torch.set_num_threads(int(no_of_threads))
    
    print(torch.get_num_threads())
    time.sleep(2)
    for x in range(1):
        n_dict={}
        n_lst=[]
        
        messages = [{"role": "user", "content": prompts[x]}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer(text, return_tensors="pt").to(model.device)
        for i in range(repeats):
            start_time=time.perf_counter()
            out_put_ids=model.generate(max_new_tokens=max_new_tokens,**model_inputs,do_sample=False)
            


            end_time=time.perf_counter()
            n_lst.append(end_time-start_time)
        mean=statistics.mean(n_lst)
        stndev=statistics.stdev(n_lst)
        n_dict['thread']=torch.get_num_threads()
        n_dict['mean']=mean
        n_dict['stdev']=stndev
        results.append(n_dict)

    
    
    
    
    
  
    
    

In [17]:
def make_prompt(item):
    lines = [item["question"]]
    for label, text in zip(item["choices"]["label"], item["choices"]["text"]):
        lines.append(f"{label}. {text}")
    lines.append("Answer with the letter and briefly explain why.")
    return "\n".join(lines)

prompts = [make_prompt(arc[i]) for i in range(10)]



In [18]:
results=[]
prompts = [make_prompt(arc[i]) for i in range(10)]
with open("prompts_arc10.json", "w") as f:
    json.dump(prompts, f, indent=2)
with open("prompts_arc10.json") as f:
    prompts = json.load(f)


In [19]:
time_counter(no_of_threads,model_hf,tokenizer,prompts,results)

12


In [20]:
results

[{'thread': 12, 'mean': 9.557667610667218, 'stdev': 0.32026349399987625}]

In [21]:
print(torch.__config__.show())

PyTorch built with:
  - GCC 4.2
  - C++ Version: 202002
  - clang 21.0.0
  - OpenMP 202011
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: DEFAULT
  - Build settings: BLAS_INFO=accelerate, BUILD_TYPE=Release, COMMIT_SHA=5c4886908584029761b579af026dcfb627c84070, CUDA_FLAGS= -DLIBCUDACXX_ENABLE_SIMPLIFIED_COMPLEX_OPERATIONS -Xfatbin -compress-all  -Wno-deprecated-gpu-targets --expt-extended-lambda -DCUB_WRAPPED_NAMESPACE=at_cuda_detail -DDISABLE_CUSPARSE_DEPRECATED -DCUDA_HAS_FP16=1 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -DC10_NODEPRECATED, CXX_COMPILER=/usr/bin/c++, CXX_FLAGS= -fvisibility-inlines-hidden -DUSE_PTHREADPOOL -DNDEBUG -DUSE_KINETO -DUSE_PYTORCH_QNNPACK -DAT_BUILD_ARM_VEC256_WITH_SLEEF -DUSE_PYTORCH_METAL_EXPORT -DSYMBOLICATE_MOBILE_DEBUG_HANDLE -DUSE_COREML_DELEGATE -O2 -fPIC -DC10_NODEPRECATED -Wall -Wextra -Werror=return-type -Werror=non-vi

In [ ]:
t1 = next(r['mean'] for r in results if r['thread'] == 1)

n_res = []
for r in results:
    n = r['thread']
    speedup = t1 / r['mean']
    efficiency = (speedup / n) * 100
    f = ((n / speedup - 1) / (n - 1)) * 100 if n > 1 else None   

    n_res.append({
        'threads': n,
        'mean': r['mean'],
        'speedup': speedup,
        'efficiency': efficiency,
        'serial_fraction': f,
    })

NameError: name 'n_results' is not defined